# E-Commerce Raw Data Validation

## Project: E-Commerce Competitive Pricing & Product Intelligence

This notebook performs an initial validation and profiling of the raw
E-Commerce dataset before data cleaning.

The purpose is to understand the quality, structure and potential
problems in the collected data.

### Data Sources

The dataset contains product information collected from:

- Amazon
- Flipkart

### Cities Covered

- Bengaluru
- Delhi
- Mumbai
- Hyderabad
- Guwahati

### Raw Dataset

Rows: 4,679

The raw dataset is intentionally not modified in this notebook.

### Objective

The objective of raw-data validation is to identify:

- Missing values
- Duplicate records
- Invalid prices
- Invalid discounts
- Invalid ratings
- Invalid review counts
- Unexpected categorical values
- Data-type problems
- Potential business duplicates
- Potential price outliers
- Coverage imbalance

The findings from this notebook will determine the cleaning rules
implemented in `src/data_cleaning.py` and demonstrated in
`02_data_cleaning.ipynb`.

## Important Principle

This notebook is for **validation and investigation**, not cleaning.

Therefore:

- We will NOT fill missing values.
- We will NOT remove duplicates.
- We will NOT remove invalid records.
- We will NOT overwrite the raw dataset.

We will only identify and document data-quality issues.

The actual transformations will be performed later in:

`src/data_cleaning.py`

In [1]:
# ================================================================
# IMPORT REQUIRED LIBRARIES
# ================================================================

import os
import numpy as np
import pandas as pd

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

pd.set_option(
    "display.max_colwidth",
    100
)

## 1. Load Raw Dataset

The raw dataset is stored in:

`data/raw/ecommerce/ecommerce_master.csv`

This is the original dataset collected from the E-Commerce platforms.

In [2]:
# ================================================================
# LOAD RAW E-COMMERCE DATA
# ================================================================

INPUT_FILE = "data/raw/ecommerce/ecommerce_master.csv"

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE}"
    )

df = pd.read_csv(
    INPUT_FILE
)

print("=" * 70)
print("RAW DATA LOADED")
print("=" * 70)

print(
    f"Rows    : {df.shape[0]}"
)

print(
    f"Columns : {df.shape[1]}"
)

RAW DATA LOADED
Rows    : 4679
Columns : 22


## 2. Dataset Structure

First, we inspect the structure of the dataset.

This helps us understand:

- Number of observations
- Number of columns
- Column names
- Data types
- Whether unexpected fields exist

In [3]:
# ================================================================
# DATASET STRUCTURE
# ================================================================

print("Dataset Shape:")
print(df.shape)

print("\nColumn Names:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i:02d}. {column}")

print("\nData Types:")

print(
    df.dtypes
)

Dataset Shape:
(4679, 22)

Column Names:
01. platform
02. product_id
03. product_name
04. brand
05. category
06. subcategory
07. model
08. variant
09. mrp
10. selling_price
11. discount_pct
12. rating
13. review_count
14. availability
15. inventory
16. seller
17. product_url
18. search_query
19. city
20. latitude
21. longitude
22. scrape_timestamp

Data Types:
platform             object
product_id           object
product_name         object
brand                object
category             object
subcategory         float64
model               float64
variant              object
mrp                 float64
selling_price       float64
discount_pct        float64
rating              float64
review_count        float64
availability           bool
inventory             int64
seller              float64
product_url          object
search_query         object
city                 object
latitude            float64
longitude           float64
scrape_timestamp     object
dtype: object


## 3. Initial Data Sample

We inspect a few records to understand what one observation
represents.

The objective is to understand the dataset's business grain before
performing validation.

In [4]:
# ================================================================
# DISPLAY SAMPLE RECORDS
# ================================================================

display(
    df.head(10)
)

,platform,product_id,product_name,brand,category,subcategory,model,variant,mrp,selling_price,discount_pct,rating,review_count,availability,inventory,seller,product_url,search_query,city,latitude,longitude,scrape_timestamp
0,Flipkart,MOBHMGHQ7MHMMHJZ,"Samsung Galaxy F70e 5G (Limelight Green, 128 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,18999.0,14999.0,21.053740,4.2,1049.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHMGHQ7MHMMHJZ,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648735
1,Flipkart,MOBHCTXH9NNZEUVH,"Samsung Galaxy M06 5G (Sage Green, 64 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,12499.0,9999.0,20.001600,4.2,108.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHCTXH9NNZEUVH,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648765
2,Flipkart,MOBHMGHHXHYZEAVB,"Samsung Galaxy F07 (Green, 64 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,11999.0,11499.0,4.167014,4.3,655.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHMGHHXHYZEAVB,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648776
3,Flipkart,MOBHZ8YSUXXHYGBR,"Samsung Galaxy F70 Pro 5G (Alpha Black, 128 GB)",Samsung,Smartphones,NaN,NaN,8 GB RAM,49999.0,29999.0,40.000800,4.3,36.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHZ8YSUXXHYGBR,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648783
4,Flipkart,MOBHMGHQF8KKB99Q,"Samsung Galaxy F70e 5G (Spotlight Blue, 128 GB)",Samsung,Smartphones,NaN,NaN,6 GB RAM,20999.0,16999.0,19.048526,4.2,760.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHMGHQF8KKB99Q,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648789
5,Flipkart,MOBH9UYFYVNKR3HR,"Samsung M06 5G (Sage Green, 128 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,13999.0,13981.0,0.128581,4.1,344.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBH9UYFYVNKR3HR,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648795
6,Flipkart,MOBHGWKEV2HYWMQP,"Samsung Galaxy M07 (Black, 64 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,11999.0,11484.0,4.292024,4.3,74.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHGWKEV2HYWMQP,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648801
7,Flipkart,MOBH9RNGFN76WMFF,"Samsung Galaxy A36 5G (Awesome Lavender, 128 GB)",Samsung,Smartphones,NaN,NaN,8 GB RAM,35999.0,26999.0,25.000694,4.3,300.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBH9RNGFN76WMFF,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648806
8,Flipkart,MOBHHQTGXCB3FHGY,"Samsung Galaxy M17e (Blitz Blue, 128 GB)",Samsung,Smartphones,NaN,NaN,4 GB RAM,16999.0,15470.0,8.994647,4.0,57.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHHQTGXCB3FHGY,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648811
9,Flipkart,MOBHMGHHMFGB82H9,"Samsung Galaxy F06 5G (Bahama Blue, 128 GB)",Samsung,Smartphones,NaN,NaN,6 GB RAM,18999.0,15499.0,18.422022,4.2,1982.0,True,1,NaN,https://www.flipkart.com/-/p/-?pid=MOBHMGHHMFGB82H9,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648816


## 4. Data Type Validation

We check whether columns have appropriate data types.

For example:

- MRP should be numeric.
- Selling price should be numeric.
- Rating should be numeric.
- Review count should be numeric.
- Latitude and longitude should be numeric.
- Availability should be boolean.
- Scrape timestamp should represent a date/time.

At this stage we only identify potential issues.
We do not change the data types.

In [5]:
# ================================================================
# DATA TYPE INSPECTION
# ================================================================

dtype_summary = pd.DataFrame({

    "column": df.columns,

    "data_type": [
        str(dtype)
        for dtype in df.dtypes
    ],

    "missing_count": [
        df[column].isna().sum()
        for column in df.columns
    ]

})

display(
    dtype_summary
)

,column,data_type,missing_count
0,platform,object,0
1,product_id,object,0
2,product_name,object,0
3,brand,object,1203
4,category,object,0
5,subcategory,float64,4679
6,model,float64,4679
7,variant,object,805
8,mrp,float64,0
9,selling_price,float64,0


## 5. Missing Value Analysis

Missing values are investigated before cleaning.

This is important because different columns require different
business decisions.

For example:

- Missing brand may be replaced with `Unknown`.
- Missing variant may be replaced with `Not Specified`.
- Missing rating should not automatically become zero.
- Missing review count should not automatically become zero.
- A column with 100% missing values may provide no analytical value.

In [6]:
# ================================================================
# MISSING VALUE ANALYSIS
# ================================================================

missing_summary = pd.DataFrame({

    "missing_count":
        df.isna().sum(),

    "missing_percentage":
        (
            df.isna()
            .mean()
            .mul(100)
            .round(2)
        )

})

missing_summary = (
    missing_summary
    .sort_values(
        "missing_count",
        ascending=False
    )
)

display(
    missing_summary
)

,missing_count,missing_percentage
subcategory,4679,100.00
model,4679,100.00
seller,4679,100.00
brand,1203,25.71
variant,805,17.20
discount_pct,53,1.13
rating,41,0.88
review_count,41,0.88
inventory,0,0.00
longitude,0,0.00


In [7]:
# ================================================================
# COLUMNS CONTAINING MISSING VALUES
# ================================================================

columns_with_missing = (
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
)

display(
    columns_with_missing
)

,missing_count,missing_percentage
subcategory,4679,100.00
model,4679,100.00
seller,4679,100.00
brand,1203,25.71
variant,805,17.20
discount_pct,53,1.13
rating,41,0.88
review_count,41,0.88


In [8]:
# ================================================================
# IDENTIFY 100% MISSING COLUMNS
# ================================================================

completely_missing_columns = [
    column
    for column in df.columns
    if df[column].isna().all()
]

print(
    "Columns with 100% missing values:"
)

print(
    completely_missing_columns
)

Columns with 100% missing values:
['subcategory', 'model', 'seller']


## 6. Exact Duplicate Validation

We check whether complete rows are repeated exactly.

An exact duplicate means every value across every column is identical.

Exact duplicates can cause double counting and should normally be
removed during cleaning.

In [9]:
# ================================================================
# EXACT DUPLICATE CHECK
# ================================================================

exact_duplicate_count = (
    df.duplicated()
    .sum()
)

print(
    f"Exact duplicate rows: {exact_duplicate_count}"
)

Exact duplicate rows: 0


## 7. Business-Level Duplicate Validation

Exact duplicates are not the only type of duplicate.

The same product can appear multiple times because the API may return
the same product for different search queries.

For this project, we investigate the following business key:

`platform + city + product_id`

This helps determine whether the same marketplace product appears
multiple times within the same city.

In [10]:
# ================================================================
# BUSINESS-LEVEL DUPLICATE CHECK
# ================================================================

business_key = [
    "platform",
    "city",
    "product_id"
]

business_duplicate_mask = (
    df.duplicated(
        subset=business_key,
        keep=False
    )
)

business_duplicate_count = (
    business_duplicate_mask.sum()
)

print(
    f"Potential business duplicate rows: "
    f"{business_duplicate_count}"
)

Potential business duplicate rows: 96


In [11]:
# ================================================================
# INSPECT BUSINESS-LEVEL DUPLICATES
# ================================================================

business_duplicates = (
    df[
        business_duplicate_mask
    ]
    .sort_values(
        business_key
    )
)

display(
    business_duplicates[
        [
            "platform",
            "city",
            "product_id",
            "product_name",
            "brand",
            "variant",
            "mrp",
            "selling_price",
            "discount_pct",
            "search_query"
        ]
    ].head(50)
)

,platform,city,product_id,product_name,brand,variant,mrp,selling_price,discount_pct,search_query
536,Amazon,Bengaluru,B0BS1RT9S2,"WH-CH520 Wireless Bluetooth Headphones On Ear with Mic, Up to 50Hrs Battery, Quick Charge, DSEE ...",NaN,NaN,5990.0,3955.0,33.973289,Sony headphones
671,Amazon,Bengaluru,B0BS1RT9S2,"WH-CH520 Wireless Bluetooth Headphones On Ear with Mic, Up to 50Hrs Battery, Quick Charge, DSEE ...",NaN,NaN,5990.0,3955.0,33.973289,JBL headphones
901,Amazon,Bengaluru,B0F84FBWQM,80 cm (32 inches) HD Smart LED TV UA32H4550FUXXL,NaN,NaN,17900.0,15490.0,13.463687,Samsung TV
969,Amazon,Bengaluru,B0F84FBWQM,80 cm (32 inches) HD Smart LED TV UA32H4550FUXXL,NaN,NaN,17900.0,15490.0,13.463687,LG TV
52,Amazon,Bengaluru,B0G81P4MPG,"Galaxy M17 5G Mobile (Sapphire Black, 6GB RAM, 128GB Storage) | 50MP OIS Triple Camera | Gorilla...",NaN,5G,23999.0,18999.0,20.834201,Samsung smartphone
252,Amazon,Bengaluru,B0G81P4MPG,"Galaxy M17 5G Mobile (Sapphire Black, 6GB RAM, 128GB Storage) | 50MP OIS Triple Camera | Gorilla...",NaN,5G,23999.0,18999.0,20.834201,Xiaomi smartphone
51,Amazon,Bengaluru,B0G81TPT89,"Galaxy M17 5G Mobile (Moonlight Silver, 6GB RAM, 128GB Storage) | 50MP OIS Triple Camera | Goril...",NaN,5G,23999.0,18999.0,20.834201,Samsung smartphone
184,Amazon,Bengaluru,B0G81TPT89,"Galaxy M17 5G Mobile (Moonlight Silver, 6GB RAM, 128GB Storage) | 50MP OIS Triple Camera | Goril...",NaN,5G,23999.0,18999.0,20.834201,OnePlus smartphone
1089,Amazon,Bengaluru,B0G8J8SKTH,"189 L, 5 Star, Digital Inverter, Direct-Cool Single Door Refrigerator (RR21H2H25CU/HL, Camellia ...",NaN,189 L,23999.0,18290.0,23.788491,LG refrigerator
1139,Amazon,Bengaluru,B0G8J8SKTH,"189 L, 5 Star, Digital Inverter, Direct-Cool Single Door Refrigerator (RR21H2H25CU/HL, Camellia ...",NaN,189 L,23999.0,18290.0,23.788491,Samsung refrigerator


## Finding

The business-level duplicate check identified repeated product
observations.

These are not necessarily data-entry errors.

The same product may have been returned for multiple search queries.

For example, a product can appear under:

- Samsung smartphone
- Xiaomi smartphone
- OnePlus smartphone

The product ID remains the same.

### Cleaning Decision

For the analytical dataset, the business grain will be:

`platform + city + product_id`

Therefore, repeated observations at this grain will be handled during
the cleaning stage.

## 8. Price Validation

Price fields are critical for this project because one of the main
business objectives is competitive pricing analysis.

We therefore validate:

- MRP
- Selling price
- Selling price versus MRP

In [12]:
# ================================================================
# PRICE VALIDATION
# ================================================================

print("MRP summary:")
display(
    df["mrp"].describe()
)

print("\nSelling price summary:")
display(
    df["selling_price"].describe()
)

MRP summary:


count    4.679000e+03
mean     5.169841e+04
std      1.564924e+05
min      0.000000e+00
25%      9.999000e+03
50%      3.299900e+04
75%      6.969500e+04
max      9.999990e+06
Name: mrp, dtype: float64


Selling price summary:


count    4.679000e+03
mean     3.726909e+04
std      5.993389e+04
min      0.000000e+00
25%      4.499000e+03
50%      2.329300e+04
75%      5.299000e+04
max      2.699990e+06
Name: selling_price, dtype: float64

In [13]:
# ================================================================
# INVALID MRP CHECK
# ================================================================

invalid_mrp = (
    df["mrp"] <= 0
)

print(
    f"Invalid MRP records: "
    f"{invalid_mrp.sum()}"
)

display(
    df[
        invalid_mrp
    ][
        [
            "platform",
            "product_id",
            "product_name",
            "mrp",
            "selling_price",
            "city"
        ]
    ].head(20)
)

Invalid MRP records: 53


,platform,product_id,product_name,mrp,selling_price,city
23,Flipkart,MOBH9AS47XHFRMJY,"Samsung Galaxy F06 5G (Bahama Blue, 128 GB)",0.0,0.0,Bengaluru
247,Amazon,B0GYWQB116,"17T (Black,12GB+256GB)|Flagship Leica Cameras|Leica UltraPure Optics|Segment Best Leica 5X Optic...",0.0,0.0,Bengaluru
264,Flipkart,COMHDZVEH5XGAYBF,HP 15 (i5 14th Gen) Intel Core 5 120U - (16 GB/512 GB SSD/Windows 11 Home) 15-fd0664TU / 15-fd06...,0.0,0.0,Bengaluru
485,Amazon,B0G92LZJD3,"Vivobook 15 (2025), 13th Gen,Intel Core i3-1315U, 8GB RAM, 512GB SSD, FHD, 15.6"", 39.6 cm, Windo...",0.0,0.0,Bengaluru
912,Amazon,B0F43CHDSN,138 cm (55 inches) Vision AI 4K Ultra HD Smart QLED TV QA55QEF1AULXL,0.0,0.0,Bengaluru
1446,Flipkart,COMH2TPS6EYHZMHF,"Acer Aspire Lite with Backlit Keyboard, Intel Core i7 12th Gen 1255U - (16 GB/512 GB SSD/Windows...",0.0,0.0,Delhi
1707,Flipkart,ACCHHKT3YM47HUBD,GDS Collapsible Bluetooth Bass Pulse Signature Sound_ZZ Bluetooth & Wired,0.0,0.0,Delhi
1708,Flipkart,ACCHHHYYXJTGG2KK,GDS Foldable Wireless Bass Boost Pro Audio_FW Bluetooth & Wired,0.0,0.0,Delhi
1709,Flipkart,ACCHGWFCGZ4ZWJYZ,"GDS Bluetooth Headphones, Foldable Wireless Design, Deep Bass Boost_R5 Bluetooth & Wired",0.0,0.0,Delhi
1710,Flipkart,ACCHGWFBXUHX9HDF,"GDS Bass-Boost Wireless Headphones with Mic & SD Card, Huge Battery_EG Bluetooth & Wired",0.0,0.0,Delhi


In [14]:
# ================================================================
# INVALID SELLING PRICE CHECK
# ================================================================

invalid_selling_price = (
    df["selling_price"] <= 0
)

print(
    f"Invalid selling price records: "
    f"{invalid_selling_price.sum()}"
)

Invalid selling price records: 53


In [15]:
# ================================================================
# SELLING PRICE GREATER THAN MRP
# ================================================================

selling_price_above_mrp = (
    df["selling_price"]
    >
    df["mrp"]
)

print(
    f"Selling price > MRP: "
    f"{selling_price_above_mrp.sum()}"
)

display(
    df[
        selling_price_above_mrp
    ][
        [
            "platform",
            "product_id",
            "product_name",
            "mrp",
            "selling_price",
            "city"
        ]
    ].head(20)
)

Selling price > MRP: 0


,platform,product_id,product_name,mrp,selling_price,city


## 9. Discount Validation

The dataset contains `discount_pct`.

We check whether discount values are logically possible.

Expected range:

`0% to 100%`

We also compare the collected discount against the price fields.

In [16]:
# ================================================================
# DISCOUNT VALIDATION
# ================================================================

print(
    df["discount_pct"].describe()
)

invalid_discount = (

    (df["discount_pct"] < 0)
    |
    (df["discount_pct"] > 100)

)

print(
    "\nInvalid discount records:",
    invalid_discount.sum()
)

count    4626.000000
mean       34.771071
std        25.546648
min         0.000000
25%        12.957471
50%        31.918629
75%        51.443270
max        97.224923
Name: discount_pct, dtype: float64

Invalid discount records: 0


In [17]:
# ================================================================
# CHECK COLLECTED DISCOUNT AGAINST CALCULATED DISCOUNT
# ================================================================

calculated_discount = (

    (
        df["mrp"]
        -
        df["selling_price"]
    )
    /
    df["mrp"]

) * 100

discount_difference = (
    df["discount_pct"]
    -
    calculated_discount
).abs()

print(
    "Maximum difference between collected and calculated discount:"
)

print(
    discount_difference.max()
)

Maximum difference between collected and calculated discount:
1.4210854715202004e-14


In [18]:
# ================================================================
# IDENTIFY SIGNIFICANT DISCOUNT DIFFERENCES
# ================================================================

discount_mismatch = (
    discount_difference > 1
)

print(
    "Records with discount difference > 1 percentage point:",
    discount_mismatch.sum()
)

Records with discount difference > 1 percentage point: 0


## 10. Rating Validation

Ratings should normally fall between:

`0 and 5`

Missing ratings are not treated as invalid because a missing rating
means the source did not provide a rating.

We therefore check only non-null ratings.

In [19]:
# ================================================================
# RATING VALIDATION
# ================================================================

print(
    df["rating"].describe()
)

invalid_rating = (

    df["rating"].notna()
    &
    (
        (df["rating"] < 0)
        |
        (df["rating"] > 5)
    )

)

print(
    "\nInvalid rating records:",
    invalid_rating.sum()
)

display(
    df[
        invalid_rating
    ][
        [
            "platform",
            "product_id",
            "product_name",
            "rating",
            "city"
        ]
    ]
)

count    4638.000000
mean        3.811298
std         1.222767
min         0.000000
25%         3.900000
50%         4.200000
75%         4.400000
max         5.000000
Name: rating, dtype: float64

Invalid rating records: 0


,platform,product_id,product_name,rating,city


## 11. Review Count Validation

Review count should never be negative.

Missing review counts are allowed because the source may not provide
this information.

In [20]:
# ================================================================
# REVIEW COUNT VALIDATION
# ================================================================

negative_review_count = (
    df["review_count"]
    .dropna()
    .lt(0)
)

print(
    "Negative review counts:",
    negative_review_count.sum()
)

print(
    "\nReview count summary:"
)

display(
    df["review_count"].describe()
)

Negative review counts: 0

Review count summary:


count      4638.000000
mean       3279.776197
std       11497.486130
min           0.000000
25%          10.000000
50%         141.000000
75%         999.000000
max      145494.000000
Name: review_count, dtype: float64

## 12. Geographic Validation

The dataset contains:

- latitude
- longitude
- city

We validate whether coordinates fall within valid geographic ranges.

Latitude:

`-90 to +90`

Longitude:

`-180 to +180`

In [21]:
# ================================================================
# LATITUDE VALIDATION
# ================================================================

invalid_latitude = (

    df["latitude"].notna()
    &
    (
        (df["latitude"] < -90)
        |
        (df["latitude"] > 90)
    )

)

print(
    "Invalid latitude records:",
    invalid_latitude.sum()
)


# ================================================================
# LONGITUDE VALIDATION
# ================================================================

invalid_longitude = (

    df["longitude"].notna()
    &
    (
        (df["longitude"] < -180)
        |
        (df["longitude"] > 180)
    )

)

print(
    "Invalid longitude records:",
    invalid_longitude.sum()
)

Invalid latitude records: 0
Invalid longitude records: 0


## 13. Platform Validation

The project is intended to contain:

- Amazon
- Flipkart

We check whether unexpected platform values are present.

In [22]:
# ================================================================
# PLATFORM VALIDATION
# ================================================================

expected_platforms = {
    "Amazon",
    "Flipkart"
}

actual_platforms = set(
    df["platform"]
    .dropna()
    .unique()
)

unexpected_platforms = (
    actual_platforms
    -
    expected_platforms
)

print(
    "Actual platforms:"
)

print(
    actual_platforms
)

print(
    "\nUnexpected platforms:"
)

print(
    unexpected_platforms
)

Actual platforms:
{'Flipkart', 'Amazon'}

Unexpected platforms:
set()


## 14. City Validation

The dataset was collected for:

- Bengaluru
- Delhi
- Mumbai
- Hyderabad
- Guwahati

We check for unexpected city values.

In [23]:
# ================================================================
# CITY VALIDATION
# ================================================================

expected_cities = {
    "Bengaluru",
    "Delhi",
    "Mumbai",
    "Hyderabad",
    "Guwahati"
}

actual_cities = set(
    df["city"]
    .dropna()
    .unique()
)

unexpected_cities = (
    actual_cities
    -
    expected_cities
)

print(
    "Actual cities:"
)

print(
    actual_cities
)

print(
    "\nUnexpected cities:"
)

print(
    unexpected_cities
)

Actual cities:
{'Bengaluru', 'Delhi', 'Hyderabad', 'Mumbai', 'Guwahati'}

Unexpected cities:
set()


## 15. Platform Distribution

We examine the number of observations collected from each platform.

This is not a cleaning operation.

The purpose is to understand whether the dataset is balanced or
imbalanced across platforms.

In [24]:
# ================================================================
# PLATFORM DISTRIBUTION
# ================================================================

platform_distribution = (
    df["platform"]
    .value_counts()
)

display(
    platform_distribution
)

platform
Flipkart    3476
Amazon      1203
Name: count, dtype: int64

In [25]:
# ================================================================
# CITY DISTRIBUTION
# ================================================================

city_distribution = (
    df["city"]
    .value_counts()
)

display(
    city_distribution
)

city
Bengaluru    1208
Delhi        1087
Guwahati     1045
Hyderabad    1043
Mumbai        296
Name: count, dtype: int64

In [26]:
# ================================================================
# PLATFORM × CITY DISTRIBUTION
# ================================================================

platform_city_distribution = pd.crosstab(
    df["city"],
    df["platform"]
)

display(
    platform_city_distribution
)

platform,Amazon,Flipkart
city,,
Bengaluru,332,876
Delhi,269,818
Guwahati,268,777
Hyderabad,273,770
Mumbai,61,235


## 16. Category Distribution

The dataset contains multiple product categories.

We inspect the distribution to identify:

- Categories represented in the dataset
- Categories with relatively few observations
- Potential collection imbalance

In [27]:
# ================================================================
# CATEGORY DISTRIBUTION
# ================================================================

category_distribution = (
    df["category"]
    .value_counts()
)

display(
    category_distribution
)

category
Smartphones           1060
Laptops                911
Headphones/Earbuds     909
Smartwatches           784
Televisions            627
Home Appliances        388
Name: count, dtype: int64

## 17. Search Query Coverage

Because the data was collected using product/search queries, it is
important to understand how many observations were generated by each
query.

This helps identify whether some products are overrepresented because
of repeated search queries.

In [28]:
# ================================================================
# SEARCH QUERY DISTRIBUTION
# ================================================================

query_distribution = (
    df["search_query"]
    .value_counts()
)

display(
    query_distribution.head(30)
)

search_query
Noise smartwatch             156
boAt earbuds                 138
OnePlus smartphone           128
Oppo smartphone              126
Realme smartphone            124
JBL headphones               122
Sony headphones              117
Samsung smartwatch           109
Sony earbuds                 108
boAt smartwatch              108
Godrej refrigerator           85
Haier refrigerator            77
Motorola smartphone           71
Xiaomi smartphone             70
Fossil smartwatch             67
Bose headphones               67
Apple MacBook                 67
Acer laptop                   67
Hisense TV                    66
iPhone                        64
Amazfit smartwatch            64
Vivo smartphone               64
Samsung smartphone            64
Google Pixel smartphone       64
Sony TV                       64
LG TV                         64
Samsung Galaxy smartphone     64
Lenovo laptop                 64
Poco smartphone               64
Realme earbuds                

## 18. Product ID Coverage

We check:

- Total rows
- Unique product IDs
- Number of repeated product IDs

This helps us understand the difference between raw observations and
unique products.

In [29]:
# ================================================================
# PRODUCT ID ANALYSIS
# ================================================================

total_rows = len(df)

unique_product_ids = (
    df["product_id"]
    .nunique()
)

repeated_product_ids = (
    total_rows
    -
    unique_product_ids
)

print(
    f"Total observations      : {total_rows}"
)

print(
    f"Unique product IDs      : {unique_product_ids}"
)

print(
    f"Repeated observations   : {repeated_product_ids}"
)

Total observations      : 4679
Unique product IDs      : 3614
Repeated observations   : 1065


## 19. Price Outlier Investigation

We use the IQR method to identify potentially unusual selling prices.

These are only **potential outliers**.

They are not automatically errors.

A very expensive laptop or television may be a legitimate product.

Therefore, the cleaning stage will flag potential outliers rather than
automatically delete them.

In [30]:
# ================================================================
# PRICE OUTLIER ANALYSIS
# ================================================================

price = (
    df["selling_price"]
    .dropna()
)

Q1 = price.quantile(
    0.25
)

Q3 = price.quantile(
    0.75
)

IQR = (
    Q3 - Q1
)

lower_bound = (
    Q1 - 1.5 * IQR
)

upper_bound = (
    Q3 + 1.5 * IQR
)

outlier_mask = (

    (df["selling_price"] < lower_bound)
    |
    (df["selling_price"] > upper_bound)

)

print(
    f"Q1: ₹{Q1:,.2f}"
)

print(
    f"Q3: ₹{Q3:,.2f}"
)

print(
    f"IQR: ₹{IQR:,.2f}"
)

print(
    f"Lower Bound: ₹{lower_bound:,.2f}"
)

print(
    f"Upper Bound: ₹{upper_bound:,.2f}"
)

print(
    f"Potential price outliers: "
    f"{outlier_mask.sum()}"
)

Q1: ₹4,499.00
Q3: ₹52,990.00
IQR: ₹48,491.00
Lower Bound: ₹-68,237.50
Upper Bound: ₹125,726.50
Potential price outliers: 185


In [31]:
# ================================================================
# INSPECT PRICE OUTLIERS
# ================================================================

price_outliers = (
    df[
        outlier_mask
    ]
    .sort_values(
        "selling_price",
        ascending=False
    )
)

display(
    price_outliers[
        [
            "platform",
            "city",
            "category",
            "brand",
            "product_name",
            "mrp",
            "selling_price"
        ]
    ].head(30)
)

,platform,city,category,brand,product_name,mrp,selling_price
2508,Flipkart,Mumbai,Televisions,TCL,TCL 291 cm (115 inch) Ultra HD (4K) Mini LED Smart Google TV with Largest Screen Ever | 20000+ L...,9999990.0,2699990.0
2505,Flipkart,Mumbai,Televisions,TCL,TCL C755 248 cm (98 inch) Ultra HD (4K) Mini LED Smart Google TV with 144Hz VRR and IMAX Enhanced,899990.0,899990.0
1669,Amazon,Delhi,Laptops,NaN,"2026 MacBook Pro Laptop with M5 Max chip with 18‑core CPU and 40‑core GPU: Built for AI, 41.05 c...",619900.0,589990.0
1673,Amazon,Delhi,Laptops,NaN,"2026 MacBook Pro Laptop with M5 Max chip with 18‑core CPU and 32‑core GPU: Built for AI, 41.05 c...",539900.0,513990.0
1629,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Max, 2026) M5 Max - (36 GB/2 TB SSD/Tahoe) MGDU4HN/A",499900.0,499900.0
1639,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Pro, 2026) M5 Pro - (24 GB/2 TB SSD/Tahoe) MGDP4HN/A",399900.0,399900.0
1541,Flipkart,Delhi,Laptops,MSI,MSI Raider 18 HX Intel Core i9 14th Gen 14900HX - (32 GB/2 TB SSD/Windows 11 Home/12 GB Graphics...,379990.0,379990.0
1657,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Pro, 2026) M5 Pro - (24 GB/1 TB SSD/Tahoe) MGEA4HN/A",359900.0,359900.0
328,Flipkart,Bengaluru,Laptops,ASUS,ASUS ExpertBook Ultra Copilot+ PC Intel Core Ultra X7 358H - (64 GB/2 TB SSD/Windows 11 Home) B9...,389990.0,352990.0
442,Flipkart,Bengaluru,Laptops,ASUS,ASUS ExpertBook Ultra Copilot+ PC Intel Core Ultra X7 358H - (64 GB/2 TB SSD/Windows 11 Home) B9...,389990.0,352990.0


## 20. Potential Data Anomalies

The validation process has identified several important issues.

### Current Findings

| Issue | Finding | Cleaning Decision |
|---|---:|---|
| Raw rows | 4,679 | Retain |
| Exact duplicates | 0 | No action required |
| Missing brand | 1,203 | Replace with `Unknown` |
| Missing variant | 805 | Replace with `Not Specified` |
| Missing rating | 41 | Preserve as missing |
| Missing review count | 41 | Preserve as missing |
| 100% missing columns | 3 | Remove |
| Invalid price records | 53 | Remove |
| Selling price > MRP | 0 | No action |
| Business duplicate rows | 96 | Deduplicate by business key |
| Invalid ratings | 0 | No action |
| Potential price outliers | 185 | Flag, do not automatically remove |
| Platforms | 2 | Valid |
| Cities | 5 | Valid |

These findings will be used to design the cleaning rules.

In [32]:
# ================================================================
# FINAL RAW DATA VALIDATION SUMMARY
# ================================================================

validation_summary = pd.DataFrame({

    "validation":
        [
            "Raw row count",
            "Raw column count",
            "Exact duplicates",
            "Business duplicate rows",
            "Missing brand",
            "Missing variant",
            "Missing rating",
            "Missing review count",
            "Completely missing columns",
            "Invalid MRP",
            "Invalid selling price",
            "Selling price > MRP",
            "Invalid discounts",
            "Invalid ratings",
            "Negative review counts",
            "Invalid latitude",
            "Invalid longitude",
            "Potential price outliers"
        ],

    "result":
        [
            len(df),
            len(df.columns),
            exact_duplicate_count,
            business_duplicate_count,
            df["brand"].isna().sum(),
            df["variant"].isna().sum(),
            df["rating"].isna().sum(),
            df["review_count"].isna().sum(),
            len(completely_missing_columns),
            invalid_mrp.sum(),
            invalid_selling_price.sum(),
            selling_price_above_mrp.sum(),
            invalid_discount.sum(),
            invalid_rating.sum(),
            negative_review_count.sum(),
            invalid_latitude.sum(),
            invalid_longitude.sum(),
            outlier_mask.sum()
        ]

})

display(
    validation_summary
)

,validation,result
0,Raw row count,4679
1,Raw column count,22
2,Exact duplicates,0
3,Business duplicate rows,96
4,Missing brand,1203
5,Missing variant,805
6,Missing rating,41
7,Missing review count,41
8,Completely missing columns,3
9,Invalid MRP,53


# Conclusion

The raw E-Commerce dataset contains 4,679 observations collected from
Amazon and Flipkart across five Indian cities.

The validation process identified several issues that must be addressed
before business analysis.

### Key Findings

1. The dataset contains no exact duplicate rows.
2. Multiple product observations occur because products can appear
   across different search queries.
3. Three columns contain 100% missing values:
   - `subcategory`
   - `model`
   - `seller`
4. Brand and variant contain missing values.
5. A small number of records contain invalid pricing information.
6. No records have selling price greater than MRP.
7. Ratings are within the expected 0–5 range.
8. Potential high-price outliers exist but may represent legitimate
   premium products.
9. Platform coverage is imbalanced between Amazon and Flipkart.
10. City coverage is also uneven, particularly for Mumbai.

### Next Step

The identified issues will be addressed in:

`02_data_cleaning.ipynb`

The automated implementation of the cleaning process is available in:

`src/data_cleaning.py`

The cleaned dataset will subsequently be independently validated using:

`03_data_quality.ipynb`